# 第12章：MoE 混合专家架构

## 本章目标
- 理解 Mixture-of-Experts (MoE) 的核心思想：用稀疏激活实现参数规模扩展
- 掌握 Router (Gating Network)、Expert FFN、Top-k routing 等关键组件
- 深入理解 DeepSeek-V2/V3 的架构创新：MLA 压缩和 Auxiliary-loss-free 负载均衡
- 从零手写 MoE Layer、辅助负载均衡损失、简化版 MLA Attention

## 前置知识
- 第1章：GPT 架构（Transformer Block、FFN、Self-Attention）
- 第11章：现代架构改进（RoPE、SwiGLU、RMSNorm 等可选）
- PyTorch `nn.Module` 基本用法

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install torch matplotlib
else:
    print("本地环境运行，请确保已安装 torch")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# 确保在 CPU 上可运行
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"使用设备: {device}")
print(f"PyTorch 版本: {torch.__version__}")

---

## 第一部分：从 Dense 到 Sparse —— 为什么需要 MoE

### Dense 模型的瓶颈

传统 Dense 模型中，每个 token 在每一层都要经过**所有参数**的计算。模型越大，每次前向传播的计算量 (FLOPs) 越大：

- 7B 模型推理 ≈ 14 GFLOPs/token（约 2 倍参数量，因为矩阵乘法）
- 70B 模型推理 ≈ 140 GFLOPs/token
- 100B Dense 模型推理 ≈ 200 GFLOPs/token

### MoE 的核心洞察

**不是每个 token 都需要用到所有参数。**

MoE 的思路：
1. 把一个巨大的 FFN 拆成 N 个小 FFN（称为 **Expert**）
2. 每个 token 只路由到 **top-k 个 Expert**（通常是 top-2）
3. 模型总参数量 = N 个 Expert 的参数之和，但计算量只等于 k 个 Expert

**FLOPs 分析示例：**

| 配置 | 总参数量 | 每次激活参数 | FLOPs/token |
|------|---------|------------|-------------|
| Dense 10B | 10B | 10B | ~20 GFLOPs |
| MoE 100B (8 Expert, top-2) | 100B | ~25B | ~50 GFLOPs |
| MoE 100B (64 Expert, top-2) | 100B | ~25B | ~50 GFLOPs |

关键：**参数量可以增长到 100B，但 FLOPs 只相当于一个 25B 的 Dense 模型。**

### 直觉类比

把 MoE 想象成一家医院：
- Dense 模型 = 每个病人都要看所有科室（效率低）
- MoE 模型 = 分诊台（Router）把病人送到最合适的 k 个科室（Expert）
- 科室越多，专长越细分，但每次只用到 k 个

参考论文：[GShard](https://arxiv.org/abs/2006.16668) (Lepikhin et al., 2020) 首次在 Transformer 中大规模使用 MoE

In [ ]:
# 演示 Dense vs MoE 的参数量和 FLOPs 差异

def analyze_flops(d_model, ff_mult=4, num_experts=8, top_k=2, seq_len=1):
    """分析 Dense FFN 和 MoE FFN 的参数量与 FLOPs"""
    ff_dim = d_model * ff_mult

    # Dense FFN: 一个大的 FFN
    dense_params = d_model * ff_dim + ff_dim * d_model  # 两层线性
    dense_flops = 2 * dense_params  # 矩阵乘法 FLOPs ≈ 2 × 参数量

    # MoE FFN: N 个小 FFN，每次只激活 top-k
    expert_params = d_model * ff_dim + ff_dim * d_model  # 每个 Expert 的参数
    moe_total_params = num_experts * expert_params  # 总参数量
    moe_active_params = top_k * expert_params  # 每次激活的参数量
    moe_flops = 2 * moe_active_params  # 只计算 top-k 个 Expert

    print(f"d_model={d_model}, ff_mult={ff_mult}, num_experts={num_experts}, top_k={top_k}")
    print(f"  Dense FFN: {dense_params/1e6:.1f}M 参数, {dense_flops/1e6:.1f}M FLOPs")
    print(f"  MoE FFN:   {moe_total_params/1e6:.1f}M 总参数, "
          f"{moe_active_params/1e6:.1f}M 激活参数, {moe_flops/1e6:.1f}M FLOPs")
    print(f"  压缩比: 参数 {moe_total_params/dense_params:.1f}x, "
          f"FLOPs 仅 {moe_flops/dense_flops:.1f}x")

print("=== 小模型示例 ===")
analyze_flops(d_model=768, ff_mult=4, num_experts=8, top_k=2)
print()
print("=== 大模型示例 (类似 Mixtral 8x7B) ===")
analyze_flops(d_model=4096, ff_mult=4, num_experts=8, top_k=2)

---

## 第二部分：Basic MoE 结构 —— Router + Expert FFN

### 核心组件

MoE Layer 由三部分组成：

1. **Router (Gating Network)**：一个线性层，输入 token embedding，输出每个 Expert 的路由分数
   - `Router: Linear(d_model, num_experts)` → `softmax` → `top-k`

2. **Expert FFN**：N 个独立的 Feed-Forward Network
   - 每个 Expert 就是一个标准的两层 FFN：`d_model → ff_dim → d_model`

3. **Forward 流程**：
   - 对每个 token 计算 Router 分数
   - 选择 top-k 个 Expert
   - 将 token 分配给选中的 Expert
   - 用 Router 权重加权组合 Expert 输出

### 数学表达

$$\text{MoE}(x) = \sum_{i \in \text{Top-k}} g_i(x) \cdot E_i(x)$$

其中 $g_i(x)$ 是 Router 对第 $i$ 个 Expert 的权重，$E_i(x)$ 是第 $i$ 个 Expert 的输出。

参考论文：[Switch Transformers](https://arxiv.org/abs/2101.03961) (Fedus et al., 2021)

In [ ]:
class Expert(nn.Module):
    """单个 Expert FFN

    结构和标准 FFN 一样：Linear → GELU → Linear
    每个 Expert 是一个独立的子网络，拥有自己的参数。
    """
    def __init__(self, d_model, ff_dim, dropout=0.0):
        super().__init__()
        self.w1 = nn.Linear(d_model, ff_dim)
        self.w2 = nn.Linear(ff_dim, d_model)
        self.act = nn.GELU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (..., d_model) -> (..., ff_dim) -> (..., d_model)
        return self.dropout(self.w2(self.act(self.w1(x))))


class TopKMoELayer(nn.Module):
    """Top-k MoE Layer

    核心组件：
    - Router: 将每个 token 映射到 num_experts 维的概率分布
    - Experts: num_experts 个独立的 FFN
    - Top-k 选择: 每个 token 只激活 top_k 个 Expert

    参考: Switch Transformers (Fedus et al., 2021)
    """
    def __init__(self, d_model, num_experts=8, top_k=2, ff_mult=4, dropout=0.0):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.d_model = d_model
        ff_dim = d_model * ff_mult

        # Router: 线性层，输出每个 Expert 的路由分数
        self.router = nn.Linear(d_model, num_experts, bias=False)

        # N 个独立的 Expert FFN
        self.experts = nn.ModuleList([
            Expert(d_model, ff_dim, dropout) for _ in range(num_experts)
        ])

    def forward(self, x):
        """
        Args:
            x: (batch_size, seq_len, d_model)
        Returns:
            output: (batch_size, seq_len, d_model)
            aux_data: dict，包含路由信息（用于后续负载均衡损失）
        """
        B, T, C = x.shape

        # ---- Step 1: 计算 Router 分数 ----
        # router_logits: (B*T, num_experts)
        router_logits = self.router(x.reshape(-1, C))

        # softmax 得到每个 Expert 的概率
        router_probs = F.softmax(router_logits, dim=-1)  # (B*T, num_experts)

        # ---- Step 2: Top-k 选择 ----
        # 选择概率最高的 top_k 个 Expert
        top_k_weights, top_k_indices = torch.topk(router_probs, self.top_k, dim=-1)
        # top_k_weights: (B*T, top_k), top_k_indices: (B*T, top_k)

        # 归一化 top-k 权重，使其和为 1
        top_k_weights = top_k_weights / top_k_weights.sum(dim=-1, keepdim=True)

        # ---- Step 3: 分配 token 到 Expert 并计算输出 ----
        x_flat = x.reshape(-1, C)  # (B*T, C)
        output = torch.zeros_like(x_flat)  # (B*T, C)

        for i in range(self.top_k):
            # 对第 i 个 top-k 选择
            expert_indices = top_k_indices[:, i]  # (B*T,) 每个 token 选中的第 i 个 Expert
            weights = top_k_weights[:, i].unsqueeze(-1)  # (B*T, 1)

            for e in range(self.num_experts):
                # 找到选择了 Expert e 的 token
                mask = (expert_indices == e)
                if mask.any():
                    expert_input = x_flat[mask]  # (num_selected, C)
                    expert_output = self.experts[e](expert_input)  # (num_selected, C)
                    output[mask] += weights[mask] * expert_output

        output = output.reshape(B, T, C)

        # 返回路由信息，用于计算辅助损失
        aux_data = {
            'router_logits': router_logits,   # 用于辅助损失
            'router_probs': router_probs,
            'top_k_indices': top_k_indices,
        }

        return output, aux_data

In [ ]:
# 测试 MoE Layer
torch.manual_seed(42)

d_model = 128
num_experts = 8
top_k = 2

moe_layer = TopKMoELayer(d_model, num_experts=num_experts, top_k=top_k, ff_mult=4)

# 统计参数量
total_params = sum(p.numel() for p in moe_layer.parameters()) / 1e6
expert_params = sum(p.numel() for p in moe_layer.experts.parameters()) / 1e6
router_params = sum(p.numel() for p in moe_layer.router.parameters()) / 1e6
print(f"MoE Layer 总参数量: {total_params:.2f}M")
print(f"  Expert 参数量: {expert_params:.2f}M ({num_experts} 个 Expert)")
print(f"  Router 参数量: {router_params:.4f}M")
print(f"  等效 Dense FFN 参数量: {expert_params/num_experts:.2f}M")
print(f"  每次激活参数量: ~{top_k * expert_params/num_experts:.2f}M (top-{top_k})")

# Forward pass
x = torch.randn(2, 16, d_model)  # batch=2, seq_len=16
output, aux = moe_layer(x)
print(f"\n输入 shape: {x.shape}")
print(f"输出 shape: {output.shape}")
print(f"Router logits shape: {aux['router_logits'].shape}")
print(f"Top-k indices shape: {aux['top_k_indices'].shape}")

# 查看路由分布
print(f"\n前5个 token 的 top-{top_k} 路由:")
for t in range(5):
    experts = aux['top_k_indices'][t].tolist()
    probs = [f"{aux['router_probs'][t, e]:.3f}" for e in experts]
    print(f"  Token {t}: Expert {experts}, 概率 {probs}")

---

## 第三部分：负载均衡问题 (Load Balancing)

### 路由坍塌 (Routing Collapse)

MoE 训练中最大的问题：**所有 token 都被 Router 分配给少数几个 Expert**，其他 Expert 被闲置。

为什么？这是一个正反馈循环：
1. 初始化时，某个 Expert 碰巧被分配了更多 token
2. 更多 token → 更多梯度 → 更新更快 → 变得更好
3. 变得更好 → Router 更倾向选择它 → 分配更多 token
4. 最终：一个 Expert 处理 80%+ 的 token，其余 Expert 形同虚设

### 辅助负载均衡损失 (Auxiliary Load Balancing Loss)

Switch Transformers 提出的经典方案：在训练损失中加入辅助损失，惩罚 Expert 间的负载不均衡。

$$\mathcal{L}_{\text{aux}} = \alpha \cdot N \cdot \sum_{i=1}^{N} f_i \cdot P_i$$

其中：
- $N$ = Expert 数量
- $f_i$ = 分配给第 $i$ 个 Expert 的 token 比例
- $P_i$ = 第 $i$ 个 Expert 的平均 Router 概率
- $\alpha$ = 辅助损失系数（通常 0.01）

**直觉**：当负载完全均衡时，$f_i = 1/N$，$P_i = 1/N$，此时 $\mathcal{L}_{\text{aux}} = \alpha$（最小值）。任何不均衡都会使损失增大。

参考：Switch Transformers Section 2.2

In [ ]:
# 演示路由坍塌问题

def show_routing_distribution(top_k_indices, num_experts):
    """展示路由分布"""
    # 统计每个 Expert 被选中的次数
    counts = torch.zeros(num_experts)
    for e in range(num_experts):
        counts[e] = (top_k_indices == e).sum().item()

    total = top_k_indices.numel()  # 总选择次数 = batch * seq_len * top_k
    fractions = counts / total

    print(f"路由分布 (理想值: {1/num_experts:.3f}):")
    for e in range(num_experts):
        bar = "█" * int(fractions[e] * 50)
        print(f"  Expert {e}: {fractions[e]:.3f} ({counts[e]:.0f}/{total}) {bar}")

    # 计算不均衡度
    ideal = 1.0 / num_experts
    imbalance = ((fractions - ideal) ** 2).sum().item()
    print(f"  不均衡度 (MSE): {imbalance:.6f}")
    return fractions

print("=== 初始化后的路由分布 (接近均匀) ===")
torch.manual_seed(42)
moe = TopKMoELayer(128, num_experts=8, top_k=2)
x = torch.randn(4, 64, 128)
_, aux = moe(x)
fracs = show_routing_distribution(aux['top_k_indices'], num_experts=8)

In [ ]:
def load_balancing_loss(router_logits, top_k_indices, num_experts):
    """辅助负载均衡损失

    L_aux = num_experts * sum(f_i * P_i)

    其中:
    - f_i = 分配给 Expert i 的 token 比例
    - P_i = Expert i 的平均 Router 概率

    当负载完全均衡时，L_aux = 1.0 (最小值)
    当负载不均衡时，L_aux > 1.0

    Args:
        router_logits: (num_tokens, num_experts) Router 原始输出
        top_k_indices: (num_tokens, top_k) 每个 token 选择的 Expert 索引
        num_experts: Expert 数量
    """
    num_tokens = router_logits.shape[0]

    # P_i: 每个 Expert 的平均 Router 概率
    router_probs = F.softmax(router_logits, dim=-1)  # (num_tokens, num_experts)
    P = router_probs.mean(dim=0)  # (num_experts,)

    # f_i: 每个 Expert 被选中的 token 比例
    # 用 one-hot 编码统计
    expert_mask = F.one_hot(top_k_indices, num_experts).float()  # (num_tokens, top_k, num_experts)
    f = expert_mask.sum(dim=(0, 1)) / (num_tokens * top_k_indices.shape[1])  # (num_experts,)

    # L_aux = num_experts * sum(f_i * P_i)
    aux_loss = num_experts * (f * P).sum()

    return aux_loss


# 测试辅助损失
print("=== 负载均衡损失测试 ===")
torch.manual_seed(42)
moe = TopKMoELayer(128, num_experts=8, top_k=2)
x = torch.randn(4, 64, 128)
_, aux = moe(x)

aux_loss = load_balancing_loss(
    aux['router_logits'], aux['top_k_indices'], num_experts=8
)
print(f"辅助损失值: {aux_loss.item():.4f}")
print(f"理论最小值 (完全均衡): {1.0:.4f}")
print(f"偏差: {(aux_loss.item() - 1.0):.4f}")

In [ ]:
# 验证：手动构造极端不均衡情况
print("=== 极端不均衡 vs 完全均衡 对比 ===")

# 极端不均衡：所有 token 都只选 Expert 0 和 Expert 1
num_tokens = 256
num_experts = 8
top_k = 2

# 构造 router_logits 让前两个 Expert 概率极高
extreme_logits = torch.zeros(num_tokens, num_experts)
extreme_logits[:, 0] = 10.0  # Expert 0 概率极高
extreme_logits[:, 1] = 5.0   # Expert 1 次之
extreme_top_k = torch.zeros(num_tokens, top_k, dtype=torch.long)
extreme_top_k[:, 0] = 0  # 全部选 Expert 0
extreme_top_k[:, 1] = 1  # 全部选 Expert 1

extreme_loss = load_balancing_loss(extreme_logits, extreme_top_k, num_experts)
print(f"极端不均衡损失: {extreme_loss.item():.4f}")

# 完全均衡：每个 Expert 被选中的比例相同
balanced_logits = torch.zeros(num_tokens, num_experts)
balanced_top_k = torch.zeros(num_tokens, top_k, dtype=torch.long)
for t in range(num_tokens):
    balanced_top_k[t, 0] = (2 * t) % num_experts
    balanced_top_k[t, 1] = (2 * t + 1) % num_experts

balanced_loss = load_balancing_loss(balanced_logits, balanced_top_k, num_experts)
print(f"完全均衡损失: {balanced_loss.item():.4f}")
print(f"\n结论: 不均衡时损失更高 ({extreme_loss.item():.4f} > {balanced_loss.item():.4f})")
print("最小化辅助损失会推动 Router 更均匀地分配 token")

---

## 第四部分：DeepSeek-V2 MLA (Multi-head Latent Attention)

### KV Cache 的瓶颈

标准 Multi-Head Attention 在推理时需要缓存所有历史 token 的 Key 和 Value：

- KV Cache 大小 = `2 × n_layers × seq_len × n_kv_heads × head_dim`
- 对于 DeepSeek-V2 (60B Active, 128K context)：KV Cache 约 **16GB** per batch
- 多用户推理时，KV Cache 成为显存瓶颈

### MLA 的核心创新

DeepSeek-V2 提出了 **Multi-head Latent Attention (MLA)**，核心思想：

1. **压缩**：将 K 和 V 投影到一个低维 latent 向量 $c_{KV}$
   - 原始 K/V 维度：$n\_kv\_heads \times head\_dim$
   - Latent 维度：$d\_latent$（远小于原始维度）

2. **缓存**：推理时只缓存 $c_{KV}$，而不是完整的 K 和 V

3. **解压**：在计算 Attention 时，从 $c_{KV}$ 恢复 K 和 V

### KV Cache 压缩比

| | 标准 MHA | MLA |
|---|---------|-----|
| 缓存内容 | K + V | $c_{KV}$ |
| 缓存维度 | $2 \times n\_kv\_heads \times head\_dim$ | $d\_latent$ |
| DeepSeek-V2 | $2 \times 128 \times 128 = 32768$ | $512$ |
| 压缩比 | 1x | **~64x** |

参考：[DeepSeek-V2](https://arxiv.org/abs/2405.04434) Section 2.1

In [ ]:
# KV Cache 大小对比
def compare_kv_cache(n_layers, seq_len, n_kv_heads, head_dim, kv_latent_dim):
    """对比标准 MHA 和 MLA 的 KV Cache 大小"""
    # 标准 MHA: 每层缓存 K 和 V
    standard_per_layer = 2 * n_kv_heads * head_dim  # 每个 token 的 K+V 参数量
    standard_total = n_layers * seq_len * standard_per_layer

    # MLA: 每层只缓存 latent 向量
    mla_per_layer = kv_latent_dim  # 每个 token 的 latent 参数量
    mla_total = n_layers * seq_len * mla_per_layer

    compression_ratio = standard_total / mla_total

    print(f"配置: {n_layers} 层, seq_len={seq_len}, "
          f"n_kv_heads={n_kv_heads}, head_dim={head_dim}, kv_latent_dim={kv_latent_dim}")
    print(f"  标准 MHA KV Cache: {standard_total/1e9:.2f} GB (fp16)")
    print(f"  MLA KV Cache:      {mla_total/1e9:.2f} GB (fp16)")
    print(f"  压缩比: {compression_ratio:.1f}x")
    print(f"  每层每个 token: 标准 {standard_per_layer} vs MLA {mla_per_layer} 参数")

print("=== DeepSeek-V2 配置 ===")
compare_kv_cache(
    n_layers=60, seq_len=128000,
    n_kv_heads=128, head_dim=128,
    kv_latent_dim=512
)

print("\n=== 小模型示例 (类似 Qwen2 配置) ===")
compare_kv_cache(
    n_layers=32, seq_len=32768,
    n_kv_heads=32, head_dim=128,
    kv_latent_dim=256
)

In [ ]:
class SimplifiedMLAAttention(nn.Module):
    """简化版 Multi-head Latent Attention (MLA)

    核心思想：
    1. 将 K, V 压缩到一个低维 latent 空间 (kv_latent_dim << n_heads * head_dim)
    2. 缓存时只存 latent 向量，大幅减少 KV Cache
    3. Attention 计算时从 latent 恢复 K, V

    注意：这是教学简化版，省略了 RoPE 等细节。
    完整实现请参考 DeepSeek-V2 论文 Section 2.1。
    """
    def __init__(self, d_model, n_heads, kv_latent_dim, dropout=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.kv_latent_dim = kv_latent_dim

        # Q 投影（和标准 Attention 一样）
        self.q_proj = nn.Linear(d_model, d_model, bias=False)

        # 压缩投影：将输入压缩到低维 latent 空间
        self.kv_compress = nn.Linear(d_model, kv_latent_dim, bias=False)

        # 解压投影：从 latent 恢复 K 和 V
        self.k_decompress = nn.Linear(kv_latent_dim, d_model, bias=False)
        self.v_decompress = nn.Linear(kv_latent_dim, d_model, bias=False)

        # 输出投影
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)

    def forward(self, x, kv_cache=None):
        """
        Args:
            x: (batch, seq_len, d_model)
            kv_cache: 可选，之前的 latent 缓存 (batch, cache_len, kv_latent_dim)
        Returns:
            output: (batch, seq_len, d_model)
            new_kv_cache: (batch, seq_len, kv_latent_dim) 新的 latent 缓存
        """
        B, T, C = x.shape

        # Step 1: 计算 Q（和标准 Attention 一样）
        q = self.q_proj(x)  # (B, T, d_model)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)  # (B, n_heads, T, head_dim)

        # Step 2: 压缩到 latent 空间（这就是 MLA 的关键！）
        latent = self.kv_compress(x)  # (B, T, kv_latent_dim) <- 比原始 K+V 小得多

        # 拼接缓存
        if kv_cache is not None:
            latent_full = torch.cat([kv_cache, latent], dim=1)  # (B, T+cache_len, kv_latent_dim)
        else:
            latent_full = latent

        # Step 3: 从 latent 恢复 K 和 V
        k = self.k_decompress(latent_full)  # (B, T_full, d_model)
        v = self.v_decompress(latent_full)  # (B, T_full, d_model)

        S = latent_full.shape[1]  # 总序列长度
        k = k.view(B, S, self.n_heads, self.head_dim).transpose(1, 2)  # (B, n_heads, S, head_dim)
        v = v.view(B, S, self.n_heads, self.head_dim).transpose(1, 2)  # (B, n_heads, S, head_dim)

        # Step 4: 标准 Attention 计算
        scale = 1.0 / math.sqrt(self.head_dim)
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) * scale  # (B, n_heads, T, S)

        # Causal mask
        causal_mask = torch.triu(
            torch.ones(T, S, device=x.device, dtype=torch.bool),
            diagonal=S - T + 1
        )
        attn_weights = attn_weights.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))
        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)

        output = torch.matmul(attn_weights, v)  # (B, n_heads, T, head_dim)
        output = output.transpose(1, 2).contiguous().view(B, T, C)
        output = self.out_proj(output)

        return output, latent  # 返回 latent 用于缓存

In [ ]:
# 测试 MLA Attention
torch.manual_seed(42)

d_model = 128
n_heads = 4
head_dim = d_model // n_heads  # 32
kv_latent_dim = 32  # 压缩到 32 维（原始 KV 需要 2 * 128 = 256 维）

mla_attn = SimplifiedMLAAttention(d_model, n_heads, kv_latent_dim)

# 对比参数量
mla_params = sum(p.numel() for p in mla_attn.parameters())
print(f"MLA 参数量: {mla_params:,}")
print(f"  Q 投影:       {d_model * d_model:,}")
print(f"  KV 压缩:      {d_model * kv_latent_dim:,} (压缩到 {kv_latent_dim} 维)")
print(f"  K 解压:       {kv_latent_dim * d_model:,}")
print(f"  V 解压:       {kv_latent_dim * d_model:,}")
print(f"  输出投影:     {d_model * d_model:,}")

# 标准 Attention 对比
standard_params = d_model * d_model * 4 + d_model * d_model  # Q, K, V, O 投影
print(f"\n标准 MHA 参数量: {standard_params:,}")
print(f"MLA 参数量:      {mla_params:,}")

# KV Cache 大小对比（每个 token）
standard_kv = 2 * d_model  # K + V
mla_kv = kv_latent_dim     # 只存 latent
print(f"\nKV Cache (每个 token):")
print(f"  标准 MHA: {standard_kv} 参数")
print(f"  MLA:      {mla_kv} 参数")
print(f"  压缩比:   {standard_kv / mla_kv:.1f}x")

# Forward pass
x = torch.randn(2, 16, d_model)
output, latent = mla_attn(x)
print(f"\n输入 shape: {x.shape}")
print(f"输出 shape: {output.shape}")
print(f"Latent shape: {latent.shape} (缓存这个，不是完整的 K+V)")

# 模拟增量推理
x_new = torch.randn(2, 1, d_model)  # 新增一个 token
output_new, latent_new = mla_attn(x_new, kv_cache=latent)
print(f"\n增量推理:")
print(f"  新输入 shape: {x_new.shape}")
print(f"  新输出 shape: {output_new.shape}")
print(f"  新 latent shape: {latent_new.shape}")
print(f"  缓存利用: 历史缓存 {latent.shape[1]} + 新增 {latent_new.shape[1]} = {latent.shape[1] + latent_new.shape[1]} tokens")

---

## 第五部分：DeepSeek-V3 Auxiliary-loss-free 负载均衡

### 辅助损失的问题

Switch Transformers 的辅助损失虽然能缓解路由坍塌，但有副作用：

1. **影响主任务质量**：辅助损失和语言建模目标不一定一致
2. **超参数敏感**：$\alpha$ 的大小难以调节，太小没用，太大会损害模型质量
3. **训练不稳定**：辅助损失的梯度可能和主损失冲突

### DeepSeek-V3 的创新：Bias-based Routing

DeepSeek-V3 提出了一种不需要辅助损失的负载均衡方法：

**核心思想**：给每个 Expert 加一个可学习的 bias 项，用于调节路由偏好。

```
# 标准 Router
scores = linear(x)          # (batch, num_experts)
weights = softmax(scores)

# DeepSeek-V3 Router (带 bias)
scores = linear(x) + bias   # bias 是可学习参数，shape: (num_experts,)
weights = softmax(scores)

# 训练时：
# - 不需要辅助损失
# - bias 自动学习调整负载均衡
# - 如果某个 Expert 负载过高，bias 会自动降低其被选中的概率
```

### 为什么 Bias 比 Auxiliary Loss 好？

| 特性 | Auxiliary Loss | Bias-based |
|------|---------------|------------|
| 对主任务的影响 | 梯度冲突 | 无冲突 |
| 超参数 | 需要 $\alpha$ | 不需要 |
| 实现复杂度 | 需要计算额外损失 | 只加一个偏置项 |
| 负载均衡速度 | 较慢 | 更快 |
| 模型质量 | 略有下降 | 更好 |

参考：[DeepSeek-V3](https://arxiv.org/abs/2412.19437) Section 2.3
源码：https://github.com/deepseek-ai/DeepSeek-V3

In [ ]:
class BiasBalancedMoELayer(nn.Module):
    """DeepSeek-V3 风格的 Bias-based 负载均衡 MoE

    与标准 MoE 的区别：
    1. Router 加了可学习的 bias 项
    2. 不需要辅助负载均衡损失
    3. bias 随训练自动调整，实现负载均衡

    参考: DeepSeek-V3 Section 2.3
    """
    def __init__(self, d_model, num_experts=8, top_k=2, ff_mult=4, dropout=0.0):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.d_model = d_model
        ff_dim = d_model * ff_mult

        # Router: 加了 bias 的版本
        self.router = nn.Linear(d_model, num_experts, bias=False)
        # 关键创新：可学习的 bias，用于负载均衡
        self.router_bias = nn.Parameter(torch.zeros(num_experts))

        # N 个 Expert FFN
        self.experts = nn.ModuleList([
            Expert(d_model, ff_dim, dropout) for _ in range(num_experts)
        ])

    def forward(self, x):
        """
        Args:
            x: (batch_size, seq_len, d_model)
        Returns:
            output: (batch_size, seq_len, d_model)
            aux_data: dict，包含路由信息
        """
        B, T, C = x.shape

        # Step 1: 计算 Router 分数（加上 bias）
        router_logits = self.router(x.reshape(-1, C))  # (B*T, num_experts)
        # 关键：加上可学习的 bias
        router_logits = router_logits + self.router_bias

        router_probs = F.softmax(router_logits, dim=-1)

        # Step 2: Top-k 选择
        top_k_weights, top_k_indices = torch.topk(router_probs, self.top_k, dim=-1)
        top_k_weights = top_k_weights / top_k_weights.sum(dim=-1, keepdim=True)

        # Step 3: 计算 Expert 输出
        x_flat = x.reshape(-1, C)
        output = torch.zeros_like(x_flat)

        for i in range(self.top_k):
            expert_indices = top_k_indices[:, i]
            weights = top_k_weights[:, i].unsqueeze(-1)

            for e in range(self.num_experts):
                mask = (expert_indices == e)
                if mask.any():
                    expert_output = self.experts[e](x_flat[mask])
                    output[mask] += weights[mask] * expert_output

        output = output.reshape(B, T, C)

        aux_data = {
            'router_logits': router_logits,
            'router_probs': router_probs,
            'top_k_indices': top_k_indices,
            'router_bias': self.router_bias.data.clone(),
        }

        return output, aux_data

In [ ]:
# 对比 Auxiliary Loss vs Bias-based 方法
torch.manual_seed(42)

d_model = 128
num_experts = 8
top_k = 2

# 方法1: 标准 MoE + 辅助损失
moe_standard = TopKMoELayer(d_model, num_experts, top_k)

# 方法2: Bias-based MoE (DeepSeek-V3 风格)
moe_bias = BiasBalancedMoELayer(d_model, num_experts, top_k)

x = torch.randn(4, 64, d_model)

# 标准 MoE
out_std, aux_std = moe_standard(x)
std_loss = load_balancing_loss(aux_std['router_logits'], aux_std['top_k_indices'], num_experts)

# Bias-based MoE
out_bias, aux_bias = moe_bias(x)

print("=== 标准 MoE (需要辅助损失) ===")
fracs_std = show_routing_distribution(aux_std['top_k_indices'], num_experts)
print(f"辅助损失值: {std_loss.item():.4f}")

print(f"\n=== Bias-based MoE (DeepSeek-V3) ===")
fracs_bias = show_routing_distribution(aux_bias['top_k_indices'], num_experts)
print(f"Router bias 值: {aux_bias['router_bias'].tolist()}")
print(f"\n注意: bias 初始化为 0，训练过程中会自动调整以实现负载均衡")
print(f"不需要额外的辅助损失，不干扰主任务梯度")

In [ ]:
# 模拟训练：观察 bias 如何自动调整实现负载均衡
torch.manual_seed(42)

d_model = 64
num_experts = 4
top_k = 2

moe = BiasBalancedMoELayer(d_model, num_experts, top_k, ff_mult=2)
optimizer = torch.optim.Adam(moe.parameters(), lr=1e-3)

# 模拟 200 步训练（只有 MoE Layer，用简单目标函数）
num_steps = 200
bias_history = []

for step in range(num_steps):
    x = torch.randn(4, 16, d_model)
    output, aux = moe(x)

    # 简单目标：让输出接近目标（模拟语言建模）
    target = torch.randn_like(output)
    loss = F.mse_loss(output, target)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 50 == 0 or step == num_steps - 1:
        bias_history.append({
            'step': step,
            'bias': aux['router_bias'].tolist(),
            'indices': aux['top_k_indices'].clone(),
        })

print("=== Router Bias 随训练变化 ===")
for record in bias_history:
    step = record['step']
    bias_vals = [f"{b:+.3f}" for b in record['bias']]
    print(f"  Step {step:3d}: bias = [{', '.join(bias_vals)}]")

# 最终路由分布
print("\n=== 训练后的路由分布 ===")
x = torch.randn(4, 64, d_model)
_, aux = moe(x)
fracs = show_routing_distribution(aux['top_k_indices'], num_experts)

---

## 第六部分：综合实验 —— 将 MoE 和 MLA 整合到 Transformer Block

### 实验目标

构建一个 mini DeepSeek-V2 风格的 Transformer Block，包含：
1. **MLA Attention**（压缩 KV Cache）
2. **MoE FFN**（稀疏激活的 Expert FFN）
3. **Bias-based 负载均衡**（DeepSeek-V3 风格）
4. **RMSNorm**（代替 LayerNorm，更高效）

然后对比 Dense Block 和 MoE Block 的参数量与计算量。

In [ ]:
class RMSNorm(nn.Module):
    """RMSNorm: LayerNorm 的高效替代

    与 LayerNorm 的区别：不做均值中心化，只除以 RMS 值。
    计算更简单，在大模型中更常用。
    """
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return self.weight * (x / rms)


class DeepSeekBlock(nn.Module):
    """DeepSeek 风格的 Transformer Block

    组件:
    - RMSNorm (Pre-norm)
    - MLA Attention (压缩 KV Cache)
    - MoE FFN with Bias-based 负载均衡
    """
    def __init__(self, d_model, n_heads, kv_latent_dim,
                 num_experts=8, top_k=2, ff_mult=4, dropout=0.0):
        super().__init__()
        self.ln1 = RMSNorm(d_model)
        self.attn = SimplifiedMLAAttention(d_model, n_heads, kv_latent_dim, dropout)
        self.ln2 = RMSNorm(d_model)
        self.moe = BiasBalancedMoELayer(d_model, num_experts, top_k, ff_mult, dropout)

    def forward(self, x, kv_cache=None):
        # Attention with residual
        attn_out, new_kv = self.attn(self.ln1(x), kv_cache)
        x = x + attn_out

        # MoE FFN with residual
        moe_out, aux = self.moe(self.ln2(x))
        x = x + moe_out

        return x, new_kv, aux

In [ ]:
# 对比 Dense Block 和 DeepSeek Block
torch.manual_seed(42)

# 配置参数
d_model = 128
n_heads = 4
n_layers = 4
num_experts = 8
top_k = 2
ff_mult = 4
kv_latent_dim = 32

# Dense Block (标准 Transformer)
class DenseBlock(nn.Module):
    def __init__(self, d_model, n_heads, ff_mult=4, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * ff_mult),
            nn.GELU(),
            nn.Linear(d_model * ff_mult, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        h = self.ln1(x)
        attn_out, _ = self.attn(h, h, h)
        x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x

# 构建 4 层模型
dense_blocks = nn.ModuleList([DenseBlock(d_model, n_heads, ff_mult) for _ in range(n_layers)])
deepseek_blocks = nn.ModuleList([
    DeepSeekBlock(d_model, n_heads, kv_latent_dim, num_experts, top_k, ff_mult)
    for _ in range(n_layers)
])

# 参数量对比
dense_params = sum(p.numel() for p in dense_blocks.parameters()) / 1e6
deepseek_params = sum(p.numel() for p in deepseek_blocks.parameters()) / 1e6

# DeepSeek Block 的激活参数量（每次只激活 top-k Expert）
# 每个 Block 的 Attention 参数 + top_k 个 Expert 的参数
single_block_active = 0
for name, p in deepseek_blocks[0].named_parameters():
    if 'experts' not in name:
        single_block_active += p.numel()
# 加上 top_k 个 Expert 的参数
expert_params_single = sum(p.numel() for p in deepseek_blocks[0].moe.experts[0].parameters())
single_block_active += top_k * expert_params_single
deepseek_active_params = n_layers * single_block_active / 1e6

print(f"=== {n_layers} 层模型对比 ===")
print(f"Dense Block 总参数量:        {dense_params:.2f}M")
print(f"DeepSeek Block 总参数量:      {deepseek_params:.2f}M")
print(f"DeepSeek Block 激活参数量:    {deepseek_active_params:.2f}M (top-{top_k} of {num_experts} Experts)")
print(f"\n总参数比:   {deepseek_params/dense_params:.2f}x")
print(f"激活参数比: {deepseek_active_params/dense_params:.2f}x")
print(f"\nKV Cache 每层每个 token:")
print(f"  Dense:    {2 * d_model} 参数 (K + V)")
print(f"  DeepSeek: {kv_latent_dim} 参数 (MLA latent)")
print(f"  压缩比:   {2 * d_model / kv_latent_dim:.1f}x")

In [ ]:
# Forward pass 对比测试
x = torch.randn(2, 32, d_model)  # batch=2, seq_len=32

# Dense forward
dense_out = x.clone()
for block in dense_blocks:
    dense_out = block(dense_out)

# DeepSeek forward
ds_out = x.clone()
all_aux = []
for block in deepseek_blocks:
    ds_out, kv, aux = block(ds_out)
    all_aux.append(aux)

print(f"输入 shape:  {x.shape}")
print(f"Dense 输出:  {dense_out.shape}")
print(f"DeepSeek 输出: {ds_out.shape}")

# 查看各层的路由分布
print(f"\n=== 各层路由分布 ===")
for layer_idx, aux in enumerate(all_aux):
    indices = aux['top_k_indices']
    counts = torch.zeros(num_experts)
    for e in range(num_experts):
        counts[e] = (indices == e).sum().item()
    fracs = counts / indices.numel()
    dist_str = " | ".join([f"E{e}:{fracs[e]:.2f}" for e in range(num_experts)])
    print(f"  Layer {layer_idx}: {dist_str}")

---

## 第七部分：论文参考与练习

### 本章要点回顾

1. **MoE 核心思想**：用 N 个 Expert FFN 替换单个大 FFN，每个 token 只激活 top-k 个 Expert，实现参数规模扩展而计算量不变
2. **负载均衡**：路由坍塌是 MoE 训练的核心问题，Switch Transformers 用辅助损失解决，DeepSeek-V3 用 bias-based 方法更优雅地解决
3. **MLA**：将 KV 压缩到低维 latent 空间，KV Cache 压缩数十倍，推理显存大幅节省
4. **DeepSeek-V2/V3**：将 MLA + MoE 结合，实现了高效的大模型架构

### 论文参考

1. **Switch Transformers** - Fedus et al., 2021  
   https://arxiv.org/abs/2101.03961  
   首次提出 top-1 routing 和辅助负载均衡损失，将 MoE 扩展到万亿参数

2. **GShard** - Lepikhin et al., 2020  
   https://arxiv.org/abs/2006.16668  
   首次在 Transformer 中大规模使用 MoE，用于多语言翻译

3. **DeepSeek-V2** - DeepSeek-AI, 2024  
   https://arxiv.org/abs/2405.04434  
   提出 MLA (Multi-head Latent Attention)，KV Cache 压缩约 100 倍

4. **DeepSeek-V3** - DeepSeek-AI, 2024  
   https://arxiv.org/abs/2412.19437  
   Auxiliary-loss-free 负载均衡，bias-based routing，更好的模型质量
   源码: https://github.com/deepseek-ai/DeepSeek-V3

### 练习

1. **修改 top-k 值**：将 `TopKMoELayer` 的 `top_k` 从 2 改为 1（Switch Transformer 风格），观察路由分布和模型输出的变化

2. **Expert 容量分析**：实现 Expert Capacity 机制——限制每个 Expert 最多处理的 token 数量。当某个 Expert 满载时，将溢出的 token 传递给其他 Expert

3. **MLA 完整实现**：在 `SimplifiedMLAAttention` 中加入 RoPE 位置编码。提示：RoPE 需要在解压后的 K 上应用，而不是在 latent 上

4. **对比实验**：在相同参数预算下（例如总共 10M 参数），对比 Dense Transformer 和 MoE Transformer 在一个小任务（如文本分类）上的训练速度和效果

5. **进阶**：阅读 DeepSeek-V3 论文 Section 2.4（Multi-Token Prediction），理解 DeepSeek-V3 如何用 MTP 进一步提升训练效率